# GCP Full Pipeline - Bronze to Silver to Gold

**The complete data engineering pipeline: ingest, clean, model, query.**

## Architecture

```mermaid
graph TD
    subgraph "Step 0: Setup"
        AUTH["Authenticate + Config"]
    end

    subgraph "Step 1: BRONZE (Raw)"
        SRC["Source Files\n(calls.json, orders.csv,\ncampaigns.csv, products.csv)"]
        GCS["GCS Bucket\n/bronze/"]
        BQ_RAW["BigQuery: raw.*\n(exact copy, no changes)"]
    end

    subgraph "Step 2: SILVER (Clean)"
        BQ_SILVER["BigQuery: silver.*\n(deduped, timezone-fixed,\ntype-corrected, nulls flagged)"]
    end

    subgraph "Step 3: GOLD (Business-Ready)"
        DIMS["Dimensions\ndim_date, dim_time,\ndim_campaign, dim_disposition"]
        FACT["Fact Table\nfact_calls"]
    end

    subgraph "Step 4: Quality Gates"
        GATES["No duplicates?\nNo orphan keys?\nRow counts match?"]
    end

    subgraph "Step 5: Reports"
        RPT["Campaign performance\nHourly volume\nConversion rates"]
    end

    AUTH --> SRC
    SRC -->|"gcloud storage cp"| GCS
    GCS -->|"bq load"| BQ_RAW
    BQ_RAW -->|"CREATE TABLE silver.* AS\nSELECT (dedup, clean)"| BQ_SILVER
    BQ_SILVER -->|"CREATE TABLE gold.dim_*"| DIMS
    BQ_SILVER -->|"CREATE TABLE gold.fact_calls\nJOIN to dimensions"| FACT
    DIMS --> FACT
    FACT --> GATES
    GATES --> RPT

    style GCS fill:#cd7f32
    style BQ_SILVER fill:#c0c0c0
    style FACT fill:#ffd700
    style DIMS fill:#ffd700
```

## Where Data Lives at Each Step

| Step | Storage | What's There |
|---|---|---|
| **Source** | Your laptop / GitHub | Raw files (JSON, CSV) |
| **Bronze** | GCS bucket + BigQuery `raw` dataset | Exact copy of source files. No changes. |
| **Silver** | BigQuery `silver` dataset | Cleaned tables. Deduped, timezone-fixed, typed. |
| **Gold** | BigQuery `gold` dataset | Star schema. Dimensions + fact table. Business-ready. |

Everything after Bronze lives in BigQuery. No intermediate buckets. This is the simplest production pattern and how most GCP-native teams work.

For very large data (terabytes+), Silver transforms would run on Dataproc (Spark) and write Parquet files to GCS before loading to BigQuery. We cover that in the Spark/Dataproc section at the end.

| Step | What It Does | BigQuery Dataset |
|---|---|---|
| **Step 0** | Setup: auth, billing, bucket, APIs | -- |
| **Step 1: Bronze** | Upload raw files to GCS, load into BigQuery | `raw` |
| **Step 2: Silver** | Dedup, timezone, types, flag nulls, unify sources | `silver` |
| **Step 3: Gold Dimensions** | Build context tables: date, time, campaign, disposition | `gold` |
| **Step 4: Gold Fact** | Build fact_calls with surrogate keys | `gold` |
| **Step 5: Quality Gates** | Validate: no dupes, no orphans, counts match | `gold` |
| **Step 6: Reports** | Business queries on the star schema | `gold` |


In [ ]:
# === CONFIGURATION ===
PROJECT_ID = "data-pipeline-project-490820"
BUCKET_NAME = f"de-accelerator-{PROJECT_ID}"
LOCATION = "us-central1"

# Three BigQuery datasets — one per layer
RAW_DATASET = "raw"           # Bronze: raw tables loaded from GCS
SILVER_DATASET = "silver"     # Silver: cleaned, deduped, timezone fixed
GOLD_DATASET = "gold"         # Gold: star schema (dimensions + facts)

print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   gs://{BUCKET_NAME}/")
print(f"Datasets: {RAW_DATASET} → {SILVER_DATASET} → {GOLD_DATASET}")

In [ ]:
# === AUTHENTICATE ===
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab.")
else:
    print("Running locally. Use 'gcloud auth login' if not authenticated.")

!gcloud config set project {PROJECT_ID}

In [ ]:
# === HELPER FUNCTION ===
# Runs BigQuery SQL and prints results.
# WHY: Colab's ! shell commands don't expand Python variables in single-quoted SQL.

import subprocess

def run_bq(sql, print_results=True):
    """Run a BigQuery SQL query. Returns stdout."""
    result = subprocess.run(["bq", "query", "--use_legacy_sql=false", "--format=prettyjson", sql],
                            capture_output=True, text=True)
    if print_results:
        print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
    return result.stdout

# Fully qualified table prefix
R = f"{PROJECT_ID}.{RAW_DATASET}"       # raw tables
S = f"{PROJECT_ID}.{SILVER_DATASET}"    # silver tables
G = f"{PROJECT_ID}.{GOLD_DATASET}"      # gold tables

print(f"Raw:    {R}.calls")
print(f"Silver: {S}.calls")
print(f"Gold:   {G}.fact_calls")

---

## Step 1: BRONZE — Upload to GCS and Load into BigQuery

Bronze = raw data, untouched. Exact copy of what the source system produced.

In [ ]:
# Get the data files (Colab: clone from GitHub)
import os
if not os.path.exists("../data") and not os.path.exists("data"):
    print("Downloading data from GitHub...")
    !git clone --depth 1 https://github.com/sunilmogadati/systems-in-production.git /tmp/de-repo 2>/dev/null
    DATA_DIR = "/tmp/de-repo/data"
elif os.path.exists("../data/calls.json"):
    DATA_DIR = "../data"
else:
    DATA_DIR = "data"

print(f"Data: {DATA_DIR}")
print(f"Files: {os.listdir(DATA_DIR)}")

In [ ]:
# Create bucket (skip if exists)
!gcloud storage buckets create gs://{BUCKET_NAME}/ --location={LOCATION} 2>&1 || echo "Bucket exists."

# Upload to Bronze layer in GCS
!gcloud storage cp {DATA_DIR}/calls.json gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/campaigns.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/orders.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/products.csv gs://{BUCKET_NAME}/bronze/

print("\nBronze layer in GCS:")
!gcloud storage ls gs://{BUCKET_NAME}/bronze/

In [ ]:
# Create RAW dataset in BigQuery
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{RAW_DATASET} 2>&1 || echo "Dataset exists."

# Load raw files into BigQuery
!bq load --autodetect --replace --source_format=NEWLINE_DELIMITED_JSON {RAW_DATASET}.calls gs://{BUCKET_NAME}/bronze/calls.json
!bq load --autodetect --replace --source_format=CSV {RAW_DATASET}.campaigns gs://{BUCKET_NAME}/bronze/campaigns.csv
!bq load --autodetect --replace --source_format=CSV {RAW_DATASET}.orders gs://{BUCKET_NAME}/bronze/orders.csv
!bq load --autodetect --replace --source_format=CSV {RAW_DATASET}.products gs://{BUCKET_NAME}/bronze/products.csv

print("\nRaw tables:")
!bq ls {RAW_DATASET}

In [ ]:
# Quick look at the raw data — what did we load?
run_bq(f"SELECT COUNT(*) AS total_calls FROM `{R}.calls`")
run_bq(f"SELECT COUNT(*) AS total_campaigns FROM `{R}.campaigns`")
run_bq(f"SELECT COUNT(*) AS total_orders FROM `{R}.orders`")

---

## Step 2: SILVER - Clean, Dedup, Unify

### How Silver Works

```mermaid
graph LR
    subgraph "BigQuery: raw (Bronze)"
        A["raw.calls\n510 rows\n(has duplicates,\nUTC timestamps,\nmixed case)"]
        B["raw.orders\n78 rows\n(has duplicates)"]
    end

    subgraph "SQL Transform"
        T["CREATE TABLE silver.* AS\nSELECT ...\n- ROW_NUMBER() dedup\n- DATETIME() timezone\n- LOWER/TRIM() standardize\n- IS NULL flag"]
    end

    subgraph "BigQuery: silver (Clean)"
        C["silver.calls\n~500 rows\n(unique call_ids,\nlocal timestamps,\nnulls flagged)"]
        D["silver.orders\n~75 rows\n(unique order_ids)"]
    end

    A --> T
    B --> T
    T --> C
    T --> D
```

**Where does Silver live?** In BigQuery, in a dataset called `silver`. Not in a GCS bucket. The SQL `CREATE OR REPLACE TABLE` reads from `raw.*` and writes to `silver.*` in one step.

**Why not a separate bucket?** For our data size, BigQuery handles both storage and compute. At terabyte scale, you would use Spark on Dataproc to transform and write Parquet files to GCS, then load into BigQuery. Same logic, different execution engine.

Silver is the **single source of truth** for all downstream consumers.

| Silver DOES | Silver Does NOT |
|---|---|
| Remove duplicates | Drop records with missing values |
| Convert timezones (UTC to local) | Join to dimension tables |
| Fix data types | Create surrogate keys |
| Standardize text (casing, whitespace) | Apply business rules |
| Flag missing values (boolean columns) | Build the star schema |


In [ ]:
# Create Silver dataset
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{SILVER_DATASET} 2>&1 || echo "Dataset exists."

In [ ]:
# === SILVER: calls ===
# Dedup by call_id, convert timezone, fix types, flag nulls, standardize text

run_bq(f"""
CREATE OR REPLACE TABLE `{S}.calls` AS

WITH cleaned AS (
    SELECT
        call_id,
        CAST(dnis AS STRING) AS dnis,
        caller_phone AS caller_ani,

        -- Timezone: convert UTC → Eastern ONCE, here in Silver
        -- Every downstream query uses these local times — no more AT TIME ZONE anywhere else
        DATETIME(start_time, 'US/Eastern') AS call_started_local,
        DATETIME(end_time, 'US/Eastern') AS call_ended_local,
        DATE(DATETIME(start_time, 'US/Eastern')) AS call_date_local,
        EXTRACT(HOUR FROM DATETIME(start_time, 'US/Eastern')) AS call_hour_local,

        duration AS duration_sec,
        LOWER(TRIM(disposition)) AS disposition,
        UPPER(TRIM(channel)) AS call_type,

        -- Flag nulls (do NOT drop — that is a Gold-layer decision)
        duration IS NULL AS has_missing_duration,
        disposition IS NULL AS has_missing_disposition,

        -- Dedup: if same call_id appears twice, keep the first by start_time
        ROW_NUMBER() OVER (PARTITION BY call_id ORDER BY start_time) AS row_num

    FROM `{R}.calls`
)

SELECT * EXCEPT(row_num)
FROM cleaned
WHERE row_num = 1
""")

print("silver.calls created.")

In [ ]:
# === SILVER: orders ===
# Dedup by order_id, fix types

run_bq(f"""
CREATE OR REPLACE TABLE `{S}.orders` AS

SELECT * EXCEPT(row_num)
FROM (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_date DESC) AS row_num
    FROM `{R}.orders`
)
WHERE row_num = 1
""")

print("silver.orders created.")

In [ ]:
# === VERIFY SILVER ===
# How many records survived cleaning? Any missing values?

print("=== Silver Verification ===")
run_bq(f"""
SELECT
    'raw.calls' AS source, COUNT(*) AS rows FROM `{R}.calls`
UNION ALL
SELECT
    'silver.calls', COUNT(*) FROM `{S}.calls`
UNION ALL
SELECT
    'raw.orders', COUNT(*) FROM `{R}.orders`
UNION ALL
SELECT
    'silver.orders', COUNT(*) FROM `{S}.orders`
""")

print("Missing values in silver.calls:")
run_bq(f"""
SELECT
    COUNT(*) AS total,
    COUNTIF(has_missing_duration) AS missing_duration,
    COUNTIF(has_missing_disposition) AS missing_disposition,
    COUNT(DISTINCT call_id) AS unique_call_ids
FROM `{S}.calls`
""")

---

### Step 2a: Incremental Loading (Production Pattern)

Everything above uses **full reload**: drop the table, recreate from scratch. That works for demos and small data. In production, you need **incremental loading**: only process NEW data since the last run.

**Why incremental matters:**

| Approach | 1,000 rows | 1,000,000 rows | 1,000,000,000 rows |
|---|---|---|---|
| Full reload | 2 seconds | 5 minutes | Hours, expensive |
| Incremental | 2 seconds | 2 seconds | 2 seconds |

Full reload reprocesses everything every time. Incremental only processes what changed. At scale, the difference is hours vs seconds and dollars vs cents.

**How incremental works in BigQuery:**

```mermaid
graph LR
    A["New raw data arrives\n(today's calls)"] --> B["Identify new rows\n(WHERE load_date > last_run)"]
    B --> C["Transform only new rows\n(dedup, clean, type)"]
    C --> D["MERGE into silver table\n(insert new, update changed)"]
```

**The key SQL pattern: MERGE**

```sql
-- Instead of CREATE OR REPLACE (full reload):
MERGE INTO silver.calls AS target
USING (
    -- Only new rows since last run
    SELECT * FROM raw.calls
    WHERE _PARTITIONTIME > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
) AS source
ON target.call_id = source.call_id
WHEN NOT MATCHED THEN INSERT (...)
WHEN MATCHED THEN UPDATE SET ...
```

**What you need for incremental:**
- A way to identify "new" rows (timestamp column, partition, or watermark)
- MERGE statement instead of CREATE OR REPLACE
- A tracking table or watermark that records "last successful run"
- Error handling: what if the same data arrives twice?

This applies to Silver AND Gold. The full reload pattern we use here is step 1. Incremental is step 2 when you move to production.


---

## Step 3: GOLD DIMENSIONS — Business Context

Dimensions provide meaning to the raw numbers. They map codes to names, dates to day-of-week, DNIS to campaigns.

| Silver | Gold Dimensions |
|:---|:---|
| Knows about data quality | Knows about business meaning |
| Changes every pipeline run | Changes rarely (new campaign, new disposition) |
| Source-agnostic | Business-aware |

Dimensions are **reference data** — maintained when the business changes, not when new call data arrives.

In [ ]:
# Create Gold dataset
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{GOLD_DATASET} 2>&1 || echo "Dataset exists."

In [ ]:
# === dim_date ===
# Pre-computed calendar. Timezone already converted.
# Every query joins on date_key (integer) and gets day_name, is_weekend, month — no parsing.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_date` AS
SELECT
    CAST(FORMAT_DATE('%Y%m%d', d) AS INT64) AS date_key,
    d AS full_date,
    FORMAT_DATE('%A', d) AS day_name,
    EXTRACT(MONTH FROM d) AS month_num,
    FORMAT_DATE('%B', d) AS month_name,
    EXTRACT(QUARTER FROM d) AS quarter,
    EXTRACT(YEAR FROM d) AS year,
    CASE WHEN EXTRACT(DAYOFWEEK FROM d) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
    DATE_TRUNC(d, ISOWEEK) AS monday_week
FROM UNNEST(GENERATE_DATE_ARRAY('2026-01-01', '2026-12-31')) AS d
""")

print("dim_date: 365 rows")

In [ ]:
# === dim_time ===
# 24 rows — one per hour. Replaces 96 half-hour columns in flat table approaches.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_time` AS
SELECT
    hour AS time_key,
    hour,
    CONCAT(LPAD(CAST(hour AS STRING), 2, '0'), ':00') AS hour_label,
    CASE
        WHEN hour BETWEEN 6 AND 11 THEN 'Morning'
        WHEN hour BETWEEN 12 AND 17 THEN 'Afternoon'
        WHEN hour BETWEEN 18 AND 21 THEN 'Evening'
        ELSE 'Night'
    END AS period
FROM UNNEST(GENERATE_ARRAY(0, 23)) AS hour
""")

print("dim_time: 24 rows")

In [ ]:
# === dim_campaign ===
# Maps DNIS (phone number) → client, campaign, channel.
# Business logic is DATA here, not code.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_campaign` AS
SELECT
    ROW_NUMBER() OVER (ORDER BY dnis) AS campaign_key,
    CAST(dnis AS STRING) AS dnis,
    campaign_name,
    client_name,
    channel,
    start_date,
    end_date
FROM `{R}.campaigns`
""")

run_bq(f"SELECT * FROM `{G}.dim_campaign`")

In [ ]:
# === dim_disposition ===
# Maps disposition text → key + boolean flags.
# Replaces CASE WHEN logic that would otherwise be in every query.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.dim_disposition` AS
SELECT
    ROW_NUMBER() OVER (ORDER BY disposition) AS disposition_key,
    disposition AS disposition_name,
    CASE WHEN disposition = 'completed' THEN TRUE ELSE FALSE END AS is_sale,
    CASE WHEN disposition IN ('voicemail', 'abandoned') THEN TRUE ELSE FALSE END AS is_abandon,
    CASE WHEN disposition = 'callback' THEN TRUE ELSE FALSE END AS is_callback
FROM (
    SELECT DISTINCT disposition
    FROM `{S}.calls`
    WHERE disposition IS NOT NULL
)
""")

run_bq(f"SELECT * FROM `{G}.dim_disposition`")

---

## Step 4: GOLD FACTS — The Join

The fact table is where Silver (clean records) meets Gold Dimensions (business context).

| Silver | Gold Dimensions | Gold Fact Table |
|:---|:---|:---|
| Clean records, one per call | Context: campaign, date, disposition | **Both combined: call record + integer dimension keys** |
| String fields (`dnis = '8005551001'`) | Surrogate keys (`campaign_key = 7`) | Integer key joins — fast, consistent |
| Rebuilt nightly | Maintained as reference data | Rebuilt nightly from Silver + Dimensions |

In [ ]:
# === fact_calls ===
# One row per call. Surrogate dimension keys. Partitioned by date. Clustered by campaign.

run_bq(f"""
CREATE OR REPLACE TABLE `{G}.fact_calls`
PARTITION BY call_date
CLUSTER BY campaign_key
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY s.call_id) AS call_key,

    -- Dimension keys (integer — fast joins)
    CAST(FORMAT_DATE('%Y%m%d', s.call_date_local) AS INT64) AS date_key,
    s.call_hour_local AS time_key,
    dc.campaign_key,
    dd.disposition_key,

    -- Source reference
    s.call_id,
    s.call_type,
    s.call_date_local AS call_date,

    -- Measures
    s.duration_sec,
    o.total AS order_total,
    o.order_id,
    CASE WHEN o.order_id IS NOT NULL THEN TRUE ELSE FALSE END AS is_order,

    -- Caller info
    s.dnis,
    s.caller_ani,

    -- Quality flags (carried from Silver)
    s.has_missing_duration,
    s.has_missing_disposition

FROM `{S}.calls` s
LEFT JOIN `{G}.dim_campaign` dc ON s.dnis = dc.dnis
LEFT JOIN `{G}.dim_disposition` dd ON s.disposition = dd.disposition_name
LEFT JOIN `{S}.orders` o ON s.call_id = o.call_id
""")

print("fact_calls created.")

---

## Step 5: VERIFY — Data Quality Gates

Before anyone queries Gold, verify the pipeline produced correct results. If any gate fails, stop and investigate.

In [ ]:
# Gate 1: No duplicate call_ids in fact table
print("Gate 1: Duplicate check")
run_bq(f"""
SELECT call_id, COUNT(*) AS dupes
FROM `{G}.fact_calls`
GROUP BY call_id HAVING COUNT(*) > 1
""")
# Expected: empty result (no duplicates)

In [ ]:
# Gate 2: Orphan dimension keys (calls without campaign or disposition match)
print("Gate 2: Orphan keys")
run_bq(f"""
SELECT
    COUNTIF(campaign_key IS NULL) AS missing_campaign,
    COUNTIF(disposition_key IS NULL) AS missing_disposition,
    COUNT(*) AS total
FROM `{G}.fact_calls`
""")
# Investigate any non-zero values — means a DNIS or disposition isn't in the dimension table

In [ ]:
# Gate 3: Row count — Silver vs Gold should match (or be close)
print("Gate 3: Row count comparison")
run_bq(f"""
SELECT 'silver.calls' AS layer, COUNT(*) AS rows FROM `{S}.calls`
UNION ALL
SELECT 'gold.fact_calls', COUNT(*) FROM `{G}.fact_calls`
""")
# Should be the same count. If Gold has fewer, some calls didn't join to any dimension.

In [ ]:
# Gate 4: Orphan orders (orders without a matching call)
print("Gate 4: Orphan orders")
run_bq(f"""
SELECT COUNT(*) AS orphan_orders
FROM `{S}.orders` o
LEFT JOIN `{G}.fact_calls` f ON o.call_id = f.call_id
WHERE f.call_key IS NULL
""")
# Non-zero = orders exist for calls that are not in the fact table. Investigate.

---

## When to Use Spark/Dataproc (Scaling Beyond BigQuery SQL)

Everything above runs as BigQuery SQL. That works well for:
- Data up to ~100GB
- Transforms expressible in SQL
- Teams comfortable with SQL

When you need Spark/Dataproc:

| Scenario | Why BigQuery SQL Isn't Enough |
|---|---|
| **Data > 100GB per load** | Spark distributes processing across many machines |
| **Complex transforms** | Python/PySpark gives more flexibility than SQL (ML feature engineering, custom parsing) |
| **Multiple output formats** | Write Parquet, Delta Lake, Iceberg (not just BigQuery tables) |
| **Cost optimization** | Process in Spark (cheap compute), load results into BigQuery (expensive storage+query) |

### How Dataproc Fits in the Pipeline

```mermaid
graph LR
    subgraph "Small Data (what we did)"
        A1["GCS Bronze"] -->|"bq load"| B1["BigQuery raw"]
        B1 -->|"SQL"| C1["BigQuery silver"]
        C1 -->|"SQL"| D1["BigQuery gold"]
    end

    subgraph "Large Data (production scale)"
        A2["GCS Bronze\n(Parquet/JSON)"] -->|"Dataproc\nPySpark job"| B2["GCS Silver\n(Parquet, partitioned)"]
        B2 -->|"bq load"| C2["BigQuery silver"]
        C2 -->|"SQL"| D2["BigQuery gold"]
    end
```

### What a Spark Silver Transform Looks Like

```python
# silver_transform.py — submit to Dataproc
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, lower, trim
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SilverTransform").getOrCreate()

# Read from Bronze in GCS
calls = spark.read.json("gs://my-bucket/bronze/calls.json")

# Same logic as our BigQuery SQL, but in PySpark
window = Window.partitionBy("call_id").orderBy("start_time")
silver_calls = (calls
    .withColumn("row_num", row_number().over(window))
    .filter(col("row_num") == 1)
    .drop("row_num")
    .withColumn("disposition", lower(trim(col("disposition"))))
)

# Write to Silver in GCS as Parquet (partitioned by date)
silver_calls.write.mode("overwrite").partitionBy("call_date").parquet(
    "gs://my-bucket/silver/calls/"
)
```

Then submit to Dataproc:
```bash
gcloud dataproc jobs submit pyspark \
    gs://my-bucket/code/silver_transform.py \
    --cluster=my-pipeline-cluster \
    --region=us-central1
```

**The thinking is identical.** Dedup, clean, standardize. The only difference is the execution engine (Spark instead of BigQuery SQL) and the output location (GCS Parquet instead of BigQuery table).

For this material, we stay with BigQuery SQL. When you need to scale, you know the Spark pattern.


---

## Step 6: QUERY — Reports on the Star Schema

The payoff. Simple joins on integer keys. No timezone logic. No dedup. No tribal knowledge.

These are the same questions from the Pandas Analytics Pipeline — now running on the star schema.

In [ ]:
# Report 1: Campaign Performance — conversion rate, revenue, duration

print("=== Campaign Performance ===")
run_bq(f"""
SELECT
    dc.client_name,
    dc.campaign_name,
    dc.channel,
    COUNT(*) AS total_calls,
    COUNTIF(f.is_order) AS orders,
    ROUND(COUNTIF(f.is_order) / COUNT(*) * 100, 1) AS conversion_pct,
    ROUND(SUM(f.order_total), 2) AS total_revenue,
    ROUND(AVG(f.duration_sec), 1) AS avg_duration
FROM `{G}.fact_calls` f
JOIN `{G}.dim_campaign` dc ON f.campaign_key = dc.campaign_key
GROUP BY dc.client_name, dc.campaign_name, dc.channel
ORDER BY total_revenue DESC
""")

In [ ]:
# Report 2: Hourly Volume — when do we need more agents?

print("=== Hourly Volume ===")
run_bq(f"""
SELECT
    dt.hour,
    dt.period,
    COUNT(*) AS total_calls,
    COUNTIF(f.is_order) AS orders,
    ROUND(AVG(f.duration_sec), 1) AS avg_duration
FROM `{G}.fact_calls` f
JOIN `{G}.dim_time` dt ON f.time_key = dt.time_key
GROUP BY dt.hour, dt.period
ORDER BY dt.hour
""")

In [ ]:
# Report 3: Day-of-Week Pattern

print("=== Day of Week ===")
run_bq(f"""
SELECT
    dd.day_name,
    dd.is_weekend,
    COUNT(*) AS total_calls,
    COUNTIF(f.is_order) AS orders,
    ROUND(COUNTIF(f.is_order) / COUNT(*) * 100, 1) AS conversion_pct
FROM `{G}.fact_calls` f
JOIN `{G}.dim_date` dd ON f.date_key = dd.date_key
GROUP BY dd.day_name, dd.is_weekend
ORDER BY total_calls DESC
""")

In [ ]:
# Report 4: Disposition Breakdown — why are calls ending the way they do?

print("=== Disposition Breakdown ===")
run_bq(f"""
SELECT
    di.disposition_name,
    di.is_sale,
    di.is_abandon,
    COUNT(*) AS call_count,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct_of_total
FROM `{G}.fact_calls` f
JOIN `{G}.dim_disposition` di ON f.disposition_key = di.disposition_key
GROUP BY di.disposition_name, di.is_sale, di.is_abandon
ORDER BY call_count DESC
""")

In [ ]:
# Report 5: The VP's Question — conversion by campaign, by day, by time period

print("=== Conversion by Campaign x Day x Time Period ===")
run_bq(f"""
SELECT
    dc.campaign_name,
    dc.channel,
    dd.day_name,
    dt.period,
    COUNT(*) AS total_calls,
    COUNTIF(f.is_order) AS orders,
    ROUND(COUNTIF(f.is_order) / COUNT(*) * 100, 1) AS conversion_pct
FROM `{G}.fact_calls` f
JOIN `{G}.dim_campaign` dc ON f.campaign_key = dc.campaign_key
JOIN `{G}.dim_date` dd ON f.date_key = dd.date_key
JOIN `{G}.dim_time` dt ON f.time_key = dt.time_key
GROUP BY dc.campaign_name, dc.channel, dd.day_name, dt.period
ORDER BY dc.campaign_name, conversion_pct DESC
""")

In [ ]:
# Report 6: Data Quality Summary

print("=== Data Quality Summary ===")
run_bq(f"""
SELECT
    COUNT(*) AS total_calls,
    COUNTIF(campaign_key IS NULL) AS no_campaign_match,
    COUNTIF(disposition_key IS NULL) AS no_disposition_match,
    COUNTIF(has_missing_duration) AS missing_duration,
    COUNTIF(has_missing_disposition) AS missing_disposition,
    COUNTIF(is_order) AS total_orders,
    ROUND(SUM(order_total), 2) AS total_revenue
FROM `{G}.fact_calls`
""")

---

## Summary — What Was Built

| Layer | Dataset | Tables | Purpose |
|:---|:---|:---|:---|
| **Bronze** | GCS bucket + `raw` | calls, campaigns, orders, products | Raw data, untouched |
| **Silver** | `silver` | calls, orders | Cleaned: deduped, timezone fixed, types correct, nulls flagged |
| **Gold Dimensions** | `gold` | dim_date, dim_time, dim_campaign, dim_disposition | Business context as data |
| **Gold Facts** | `gold` | fact_calls | One row per call, dimension keys, measures |

```
Source Systems → GCS Bronze → BigQuery Raw → Silver (clean) → Gold Dimensions + Gold Facts → Reports
```

Each layer has one job. A bug in one layer does not cascade to the others.

In [ ]:
# Final count summary
print("=== Pipeline Complete ===")
for dataset, table in [(R, 'calls'), (R, 'campaigns'), (R, 'orders'),
                        (S, 'calls'), (S, 'orders'),
                        (G, 'dim_date'), (G, 'dim_time'), (G, 'dim_campaign'),
                        (G, 'dim_disposition'), (G, 'fact_calls')]:
    result = subprocess.run(
        ["bq", "query", "--use_legacy_sql=false", "--format=csv",
         f"SELECT COUNT(*) as c FROM `{dataset}.{table}`"],
        capture_output=True, text=True
    )
    count = result.stdout.strip().split('\n')[-1] if result.returncode == 0 else 'ERROR'
    print(f"  {dataset}.{table}: {count} rows")

In [ ]:
# === CLEANUP (uncomment to delete everything) ===
# !bq rm -r -f {RAW_DATASET}
# !bq rm -r -f {SILVER_DATASET}
# !bq rm -r -f {GOLD_DATASET}
# !gcloud storage rm -r gs://{BUCKET_NAME}/
# print("All resources deleted.")